# Wine Quality - Notebook Oficial (Binário)

Este notebook documenta e executa o **mesmo fluxo** dos scripts em `src/` para classificação binária:
- `0 = Not Good` (`quality < 7`)
- `1 = Good` (`quality >= 7`)

Pipeline oficial: `ingest -> preprocess -> prepare -> train -> evaluate`.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/wine_quality.csv')
print('Shape:', df.shape)
print('quality raw:')
print(df['quality'].value_counts().sort_index())

y_bin = (df['quality'] >= 7).astype(int)
print('\nquality_binary (0=not_good, 1=good):')
print(y_bin.value_counts().sort_index())

## Executar o pipeline dos scripts

Esta célula executa exatamente os mesmos scripts usados em produção e no DVC.

In [ ]:
import subprocess

steps = [
    ['python3', '../src/ingestion.py'],
    ['python3', '../src/preprocessing.py'],
    ['python3', '../src/prepare_data.py'],
    ['python3', '../src/train.py'],
    ['python3', '../src/evaluate.py'],
]

for step in steps:
    print('>>', ' '.join(step))
    result = subprocess.run(step, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Falha no step: {step}')

## Ler métricas oficiais geradas

As métricas abaixo vêm dos artefatos `reports/training_report.json` e `reports/evaluation_report.json`.

In [ ]:
import json
import pandas as pd

train_report = json.load(open('../reports/training_report.json'))
eval_report = json.load(open('../reports/evaluation_report.json'))

train_df = pd.DataFrame(train_report).T
print('Training (val):')
display(train_df[['val_binary_accuracy','val_binary_f1_weighted','val_binary_f1_weighted_tuned','threshold_binary_t']].sort_values('val_binary_f1_weighted_tuned', ascending=False))

eval_rows = {k:v for k,v in eval_report.items() if isinstance(v, dict) and 'test_binary_f1_weighted' in v}
eval_df = pd.DataFrame(eval_rows).T
print('\nEvaluation (test):')
display(eval_df[['test_binary_accuracy','test_binary_f1_weighted','threshold_binary_t']].sort_values('test_binary_f1_weighted', ascending=False))